<a href="https://colab.research.google.com/github/yrhutu21/NLP/blob/main/Simple_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim



In [ ]:
corpus = [
    "I love machine learning",
    "wordvec is great algorithm",
    "Implementing word2vec is fun"
]

In [ ]:
sentences = [s.lower().split() for s in corpus]

In [27]:
sentences

[['i', 'love', 'machine', 'learning'],
 ['wordvec', 'is', 'great', 'algorithm'],
 ['implementing', 'word2vec', 'is', 'fun']]

In [28]:
vocab = sorted(set(word for sent in sentences for word in sent))


In [29]:
vocab

['algorithm',
 'fun',
 'great',
 'i',
 'implementing',
 'is',
 'learning',
 'love',
 'machine',
 'word2vec',
 'wordvec']

In [ ]:
word2idx = {w:i for i,w in enumerate(vocab)}
#idx2word = {i:w for w,i in word2idx.items()}
idx2word = {i:w for i,w in enumerate(vocab)}

In [ ]:
idx2word

{0: 'algorithm',
 1: 'fun',
 2: 'great',
 3: 'i',
 4: 'implementing',
 5: 'is',
 6: 'learning',
 7: 'love',
 8: 'machine',
 9: 'word2vec',
 10: 'wordvec'}

In [30]:
vocab_size = len(vocab)

In [31]:
word2idx

{'algorithm': 0,
 'fun': 1,
 'great': 2,
 'i': 3,
 'implementing': 4,
 'is': 5,
 'learning': 6,
 'love': 7,
 'machine': 8,
 'word2vec': 9,
 'wordvec': 10}

In [32]:
X = []
Y = []
for sent in sentences:
    for i in range(len(sent)-1):
        X.append(word2idx[sent[i]])
        Y.append(word2idx[sent[i+1]])

In [45]:
X = torch.tensor(X)
Y = torch.tensor(Y)

/tmp/ipykernel_4015/3876772689.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X)
/tmp/ipykernel_4015/3876772689.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Y = torch.tensor(Y)


In [46]:
print(X)
print(Y)

tensor([ 3,  7,  8, 10,  5,  2,  4,  9,  5])
tensor([7, 8, 6, 5, 2, 0, 9, 5, 1])


In [47]:
vocab_size

11

In [55]:
class NextWordRNN(nn.Module):

    def __init__(self, vocab_size, embed_dim=10, hidden_dim=16):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):

        x = self.embedding(x)
        #print(x.shape)
        out, hx = self.rnn(x)
        #print(out.shape,hx.shape)
        out = self.fc(out[:, -1, :])
        #print(out.shape)

        return out


In [56]:
model = NextWordRNN(vocab_size)

In [36]:
 #inputs = X.unsqueeze(1)
 #outputs = model(inputs)

In [38]:
#torch.Size([9, 1, 100])
#torch.Size([9, 1, 16])
#torch.Size([9, 16])
#torch.Size([9, 10])

torch.Size([9, 10])

In [57]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [58]:

for epoch in range(300):

    optimizer.zero_grad()

    inputs = X.unsqueeze(1)
    outputs = model(inputs)

    loss = loss_fn(outputs, Y)

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print("Epoch:", epoch, "Loss:", loss.item())

Epoch: 0 Loss: 2.381201982498169
Epoch: 50 Loss: 0.18666888773441315
Epoch: 100 Loss: 0.1633411943912506
Epoch: 150 Loss: 0.1593126654624939
Epoch: 200 Loss: 0.15750986337661743
Epoch: 250 Loss: 0.15653038024902344


In [59]:
def predict_next(word):
    model.eval()
    idx = torch.tensor([[word2idx[word.lower()]]])

    with torch.no_grad():
        out = model(idx)
        pred = torch.argmax(out).item()

    return idx2word[pred]

In [60]:
print("machine=>", predict_next("machine"))
print("is=>", predict_next("is"))
print("word2vec=>", predict_next("word2vec"))

machine=> learning
is=> fun
word2vec=> is
